# 3 · Faithful ports and the corrected model

`znmno2_model.faithful` reproduces the original research programs, defects included (`docs/deviations.md`). This notebook runs the original **charge line** on an illustrative cell and compares it with the corrected model set up two ways:

- **"original treatments"**: the corrected model with the options that match the original's treatments (no transported H⁺, pH from ZHS solubility, totals in the Nernst terms, ZHS lumped into R2, the empirical OCP without Nernstian ends, no R1) and the same geometry; the reservoir is a probe region of the same volume;
- **default**: the corrected model with its defaults (speciation pH, transported H⁺, kinetic ZHS, Nernstian ends on the OCP).

The rate constants and equilibrium potentials are written differently in the two formulations (`docs/model.md`), so the curves are not expected to coincide; the comparison shows the behaviour, not a calibration. The faithful run takes a few minutes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from znmno2_model.params import Params
from znmno2_model.simulate import run
from znmno2_model.model import Model
from znmno2_model.faithful.charge import ChargeModel
from znmno2_model.faithful.constants import Template

## The original charge line

The original reads its inputs as text substituted into the source; here they are an illustrative row. Without the private copy of the original BAND/MATINV code, the solve uses bandsolver's legacy pivot: the output agrees with the original program's to about 10⁻⁹ relative.

In [ ]:
t = Template(RXNK_2="-8.5", RXNK_3="-7.5", FRAC_ZMCX="0.4", FRAC_ZMCMAX="0.03", XMAX="0.022",
             APPLIED_CURRENT="0.000121", POROSITY="0.8", VOLFRAC_MNO2="0.07", STATED_MASS_LOADING="0.00121",
             ZHS_KSP="30")
fm = ChargeModel(t)
print(fm.run())
cols = fm.rows[0].split()
fa = np.array([[float(v) for i, v in enumerate(r.split()) if i != 4] for r in fm.rows[1:]])
fcol = lambda name: fa[:, [c for c in cols if c != "State"].index(name)]

## The corrected model on the same cell

Same cathode thickness, porosity, solid fractions, current (100 mA/g) and electrolyte; the original's 100 µL reservoir becomes a probe region of that volume.

In [ ]:
reservoir_cm = 1e-4 / 0.178134094          # 100 uL over the cell area
cell = dict(L_cath=0.022, eps_cath=0.8, vf_ZMO=0.07 * 0.03, vf_host=0.07 * 0.4, vf_MnO2=1e-4, mass_AM=0.00121,
            L_probe=reservoir_cm, n_probe=40, steps="cc I=100 Vmin=0.8")
original_like = dict(species="no_H", ph_mode="zhs_equilibrium", basis="totals", zhs="lumped", r3_ocp="spline",
                     R1_on=False, logK_ZHS=30.0)
r_orig = run(Params(**cell, **original_like))
r_def = run(Params(**cell))
for lab, r in (("original treatments", r_orig), ("default", r_def)):
    print(f"{lab:22s} {r.exit_reason:14s} {r.column('mAhg')[-1]:6.1f} mAh/g")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].plot(fcol("mAh/g"), fcol("Voltage"), "k", label="faithful (original charge line)")
ax[0].plot(r_orig.column("mAhg"), r_orig.column("V"), label="corrected, original treatments")
ax[0].plot(r_def.column("mAhg"), r_def.column("V"), label="corrected, default")
ax[0].set(xlabel="capacity [mAh/g]", ylabel="V", ylim=(0.8, 1.9)); ax[0].legend(fontsize=8)
ax[1].plot(fcol("mAh/g"), fcol("pH_ZHS(avg)"), "k", label="faithful: pH used (ZHS closed form)")
ax[1].plot(r_orig.column("mAhg"), r_orig.column("pH_cath"), label="corrected, original treatments (solution pH)")
ax[1].plot(r_def.column("mAhg"), r_def.column("pH_cath"), label="corrected, default")
ax[1].set(xlabel="capacity [mAh/g]", ylabel="pH"); ax[1].legend(fontsize=8);

Things to notice:

- The faithful line's pH is the ZHS solubility pH everywhere and always (with its +1/6 exponent and K_sp = 10³⁰), not the solution's pH (deviation M-10).
- Its Mn²⁺ is uncharged (M-1) and its solids are updated explicitly with clipped rates (M-5, M-9), so it does not conserve S and Mn (M-15).
- The corrected model with the original's treatments conserves everything and reverses on charge; the remaining differences from the faithful line are the formulation of the rates and potentials.

The pH-cell line (`faithful.phcell`) runs the same way (`PhCellModel(PhTemplate(RXNK_5=..., FRACTION_KMNO2=...))`); a full run takes about 20 minutes.